# 04 — Canonical Parquet and equivalent LaminDB queries

Canonical Parquet is Jouvence's working data plane. LaminDB is a queryable registry/catalog mirror at the exact instance `jkobject/jouvencekb`, but its ingestion is currently partial and external access is blocked pending repair and grants. This notebook remains useful in fixture mode and makes live Lamin access a separate read-only opt-in.


In [ ]:
from __future__ import annotations
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from manage_db.public_notebooks import (
    PUBLIC_KG_ROOT,
    build_public_fixture,
    parquet_catalog,
    read_bounded_parquet,
)

MODE = os.environ.get("JOUVENCE_DATA_MODE", "fixture")
BILLING_PROJECT = os.environ.get("JOUVENCE_BILLING_PROJECT")
CACHE = Path(os.environ.get("JOUVENCE_NOTEBOOK_CACHE", REPO_ROOT / "artifacts" / "cache" / "public-notebooks"))
CACHE.mkdir(parents=True, exist_ok=True)
KG_ROOT = build_public_fixture(CACHE / "kg-fixture") if MODE == "fixture" else PUBLIC_KG_ROOT
print({"mode": MODE, "kg_root": str(KG_ROOT), "bounded": True})


## Separate the canonical data plane from the catalog mirror

Parquet under the reviewed canonical root holds node, edge, evidence, metadata, feature, and optional proof objects. LaminDB indexes exact entities and generic edge/evidence records for discovery. It does not redefine canonical truth or make incomplete ingestion equivalent to absence.


In [ ]:
# Separate the canonical data plane from the catalog mirror: run the bounded example and inspect its typed output.
comparison = pd.DataFrame([
    ("canonical Parquet", "reviewed object paths", "working data plane", "bounded named reads"),
    ("LaminDB", "jkobject/jouvencekb", "partial catalog mirror", "exact-ID/filter queries"),
], columns=["surface", "identity", "role", "public pattern"])
display(comparison)


### Interpretation

A Lamin row should correspond to canonical identity and provenance, but current coverage is partial. Parquet remains the fallback for truth when a Lamin query is empty or access is unavailable.


In [ ]:
catalog = parquet_catalog(KG_ROOT, billing_project=BILLING_PROJECT)
display(catalog.groupby("layer").agg(tables=("path", "count"), rows_in_selected_root=("rows", "sum")))


### Checkpoint

Fixture catalog totals are synthetic. Live catalog discovery should use committed inventories or named objects rather than an all-relation remote scan.


## Configure only the exact read-only Lamin instance

The accepted instance slug is `jkobject/jouvencekb`. Live access requires a caller's own LaminHub account and an explicit `JOUVENCE_LAMIN_LIVE=1`; the helper verifies the connected slug before querying and exposes no create, update, delete, or sync operations.


In [ ]:
# Configure only the exact read-only Lamin instance: run the bounded example and inspect its typed output.
from manage_db.public_notebooks import LAMIN_INSTANCE
LIVE_LAMIN = os.environ.get("JOUVENCE_LAMIN_LIVE", "0") == "1"
print({"required_instance": LAMIN_INSTANCE, "live_opt_in": LIVE_LAMIN, "write_capability": False})


### Interpretation

At present, a fresh external collaborator is blocked because authenticated instance/storage access is not yet reproducible. `lamin login` alone is not proof that the remote database object and permissions work.


In [ ]:
future_read_only_setup = [
    "uv run lamin login",
    "uv run lamin connect jkobject/jouvencekb",
    "export JOUVENCE_LAMIN_LIVE=1",
]
print("Run only after the maintainer confirms repaired read access:\n" + "\n".join(future_read_only_setup))


### Checkpoint

Never connect to a similarly named instance, run `lamin disconnect` as a workaround, re-upload a database, or write registries from this public course.


## Perform the canonical Parquet exact-ID lookup

We first answer through the data plane. The local fixture helper joins TP53 association assertions, disease labels, and evidence. This result is the reference for understanding the equivalent registry query shape.


In [ ]:
# Perform the canonical Parquet exact-ID lookup: run the bounded example and inspect its typed output.
from manage_db.public_notebooks import diseases_with_gene_evidence
if MODE == "fixture":
    parquet_answer = diseases_with_gene_evidence(KG_ROOT, "ENSG00000141510", limit=20)
else:
    parquet_answer = pd.DataFrame()
    print("Use an explicitly staged bounded subset for the local exact-ID helper.")
display(parquet_answer)


### Interpretation

The query is exact-ID and bounded. It does not search aliases, fuzzy labels, or every relation. Expand the biological question deliberately rather than widening the storage scan.


In [ ]:
parquet_contract = {
    "gene_id": "ENSG00000141510",
    "relation": "disease_associated_gene",
    "limit": 20,
    "canonical_truth": True if MODE == "live" else "synthetic fixture",
}
print(parquet_contract)


### Checkpoint

Record the exact relation and identifier so an equivalent Lamin query can be compared semantically rather than only by row count.


## Express equivalent node, edge, and evidence filters in LaminDB

LaminDB uses typed node registries and generic KG edge/evidence registries. The public helpers constrain exact node IDs, one relation, optional typed endpoints, deterministic ordering, and a row limit.


In [ ]:
# Express equivalent node, edge, and evidence filters in LaminDB: run the bounded example and inspect its typed output.
from manage_db.public_notebooks import query_lamindb_edges, query_lamindb_evidence, query_lamindb_node
if LIVE_LAMIN:
    lamin_node = query_lamindb_node("gene", "ENSG00000141510", limit=1)
    lamin_edges = query_lamindb_edges(relation="disease_associated_gene", x_id="ENSG00000141510", limit=20)
    lamin_evidence = query_lamindb_evidence(relation="disease_associated_gene", x_id="ENSG00000141510", limit=50)
else:
    lamin_node = lamin_edges = lamin_evidence = pd.DataFrame()
    print("Lamin query skipped; fixture mode needs no LaminHub account.")


### Interpretation

The node lookup validates the appropriate stable-ID field for each registry. Edge and evidence filters retain relation and endpoints. They are equivalent in scientific intent to the Parquet question, not necessarily in current row coverage.


In [ ]:
print({"node_rows": len(lamin_node), "edge_rows": len(lamin_edges), "evidence_rows": len(lamin_evidence)})
if LIVE_LAMIN:
    display(lamin_node); display(lamin_edges.head()); display(lamin_evidence.head())


### Checkpoint

An empty result is a query outcome, not a biological conclusion. Check mirror coverage and canonical Parquet before asserting absence.


## Compare identities while tolerating partial coverage

When both surfaces are available, compare relation and typed endpoint identities—not database primary keys or row order. Differences should be classified as expected partial ingestion, query mismatch, stale mirror, or a true integrity problem requiring review.


In [ ]:
# Compare identities while tolerating partial coverage: run the bounded example and inspect its typed output.
identity_columns = ["relation", "x_id", "x_type", "y_id", "y_type"]
comparison_plan = {
    "keys": identity_columns,
    "expected_current_state": "partial LaminDB coverage",
    "absence_rule": "Lamin-empty never proves canonical absence",
}
print(json.dumps(comparison_plan, indent=2))


### Interpretation

Do not hide mismatch by filling synthetic rows or promoting physical counts as accepted coverage. A complete parity claim requires a separately reviewed exact-ID audit with mismatch zero.


In [ ]:
if LIVE_LAMIN and not lamin_edges.empty and not parquet_answer.empty:
    lamin_disease_ids = set(lamin_edges["y_id"].astype(str))
    parquet_disease_ids = set(parquet_answer["disease_id"].astype(str))
    print({"only_parquet": sorted(parquet_disease_ids - lamin_disease_ids), "only_lamin": sorted(lamin_disease_ids - parquet_disease_ids)})
else:
    print("Cross-surface parity not claimed in this environment.")


### Checkpoint

Label the outcome precisely: fixture demonstration, bounded live lookup, partial mirror result, or blocked access—never simply “database complete.”


## Troubleshoot safely and choose the truthful fallback

Separate LaminHub authentication, instance authorization, storage/database-object availability, schema version, and mirror coverage. A failure at one layer should not trigger writes, a new instance, or a broad canonical scan.


In [ ]:
# Troubleshoot safely and choose the truthful fallback: run the bounded example and inspect its typed output.
lamin_troubleshooting = pd.DataFrame([
    ("login required", "authenticate with caller account", "no shared token"),
    ("instance denied", "request read access", "do not change slug"),
    ("database object missing", "maintainer repair required", "do not re-upload"),
    ("empty query", "compare bounded canonical Parquet", "mirror may be partial"),
    ("schema mismatch", "report versions and query", "do not mutate registries"),
], columns=["symptom", "next check", "safety boundary"])
display(lamin_troubleshooting)


### Interpretation

The truthful fallback is canonical Parquet through bounded named reads. Fixture mode remains the universally executable teaching path while live external Lamin access is blocked.


In [ ]:
print({
    "fallback": "canonical Parquet",
    "current_lamin_status": "partial and external-blocked",
    "next_notebook": "05_sampled_pyg_heterodata.ipynb",
    "canonical_writes": False,
})


### Checkpoint

Continue to Notebook 05 to see how bounded Parquet entities and assertions become a typed PyG sample without claiming a production/full export.
